In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# Configuramos Pandas para que nos muestre todas las columnas sin esconderlas
pd.set_option('display.max_columns', None)
print("Librerías importadas correctamente.")

In [ ]:
print("Cargando los Shapefiles en memoria...")
# Cargamos el mapa de ruido (Polígonos)
gdf_ruido = gpd.read_file("mapa_de_ruido_diurno.shp")
gdf_ruido.columns = gdf_ruido.columns.str.lower()
gdf_ruido = gdf_ruido.to_crs(epsg=4326)

# Cargamos el mapa de propiedades (Puntos)
gdf_deptos = gpd.read_file("Departamentos_venta_2018.shp") 
gdf_deptos.columns = gdf_deptos.columns.str.lower()
gdf_deptos = gdf_deptos.to_crs(epsg=4326)

print(f"Cargados: {len(gdf_deptos)} departamentos y {len(gdf_ruido)} polígonos acústicos.")

In [ ]:
print("Ejecutando el cruce espacial (Spatial Join)...")
# Usamos 'inner' para descartar deptos que estén fuera de CABA o en zonas sin datos
deptos_con_ruido = gpd.sjoin(gdf_deptos, gdf_ruido, how="inner", predicate="within")

print(f"¡Cruce exitoso! Quedaron {len(deptos_con_ruido)} departamentos con datos acústicos asignados.")

In [ ]:
# Miramos qué columnas trae ahora nuestro dataset
print("Columnas disponibles:")
print(deptos_con_ruido.columns.tolist())

# Imprimimos las primeras 3 filas para ver cómo se ven los datos crudos
deptos_con_ruido.head(3)

In [ ]:
# ==========================================
# Limpieza y Feature Engineering
# ==========================================
print("Iniciando limpieza del dataset cruzado...")

# 1. Hacemos una copia para no alterar el DataFrame original cruzado
df_limpio = deptos_con_ruido.copy()

# 2. FILTRADO DE VALORES NULOS Y OUTLIERS BÁSICOS
# solo propiedades que tienen precio en dólares y metros cuadrados
df_limpio = df_limpio.dropna(subset=['preciousd', 'm2total', 'db_lo'])

# Filtramos propiedades irreales
# y precios extremadamente bajos o altos
df_limpio = df_limpio[(df_limpio['m2total'] >= 15) & (df_limpio['m2total'] <= 350)]
df_limpio = df_limpio[(df_limpio['preciousd'] >= 20000) & (df_limpio['preciousd'] <= 2000000)]

# 3. CREACIÓN DE VARIABLES (Feature Engineering)
# Precio por Metro Cuadrado
df_limpio['precio_x_m2'] = df_limpio['preciousd'] / df_limpio['m2total']

# Aseguramos que la columna de ruido (db_lo) sea numérica para que el algoritmo pueda leerla
df_limpio['db_lo_num'] = pd.to_numeric(df_limpio['db_lo'], errors='coerce')
df_limpio = df_limpio.dropna(subset=['db_lo_num'])

# 4. SELECCIÓN DE COLUMNAS FINALES PARA EL MODELO
# Descartamos texto libre 
columnas_modelo = [
    'barrio_1',      # Necesario para entender el peso de la zona (ej. Puerto Madero vs. Lugano)
    'ambientes',     # Tamaño funcional
    'm2total',       # Tamaño físico
    'db_lo_num',     # El factor ambiental (Nivel de ruido)
    'preciousd',     # Precio total (Target opcional)
    'precio_x_m2',   # Nuestro Target (Variable a predecir) principal
    'latitud',       # Coordenada Y
    'longitud'       # Coordenada X
]

df_modelo = df_limpio[columnas_modelo].copy()

# 5. GUARDADO DEL DATASET LIMPIO
# Lo guardamos como CSV
df_modelo.to_csv("dataset_inmobiliario_acustico_limpio.csv", index=False)

print(f"Limpieza finalizada. Dataset original: {len(deptos_con_ruido)} filas. Dataset limpio: {len(df_modelo)} filas.")
print("Primeras 3 filas listas para el algoritmo:")
df_modelo.head(3)